In [1]:
import os

In [2]:
%pwd

'c:\\Users\\QSUS\\Desktop\\6_sem\\course\\EDEP\\DS_edep\\research'

In [3]:
os.chdir("../")
%pwd

'c:\\Users\\QSUS\\Desktop\\6_sem\\course\\EDEP\\DS_edep'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [5]:
from src.ds_edep.constants import *
from src.ds_edep.utils.common import read_yaml, create_directories

[2026-06-05 22:34:56,216: INFO: __init__: Logger setup completed!]


In [6]:
class ConfigManager:
    def __init__(self,
                 config_path = CONFIG_FILE_PATH,
                 params_path = PARAMS_FILE_PATH,
                 schema_path = SCHEMA_FILE_PATH):
        self.config_path = read_yaml(config_path)
        self.params_path = read_yaml(params_path)
        schema_path = read_yaml(schema_path)
        
        create_directories([self.config_path.artifacts_root])
        
    def get_data_ingestion_config(self):
        config = self.config_path.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config=DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir

        )
        return data_ingestion_config
        

In [7]:
import os
import urllib.request as request
import zipfile
from src.ds_edep import main_logger

In [ ]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config
        
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            main_logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            main_logger.info(f"File already exists")
            
    def unzip_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, "r") as zip_file:
            zip_file.extractall(unzip_path)

In [9]:
try:
    config=ConfigManager()
    data_ingestion_config=config.get_data_ingestion_config()
    data_ingestion=DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.unzip_zip_file()
except Exception as e:
    raise e

[2026-06-05 22:34:56,945: INFO: common: created directory at: artifacts]
[2026-06-05 22:34:56,948: INFO: common: created directory at: artifacts/data_ingestion]


[2026-06-05 22:34:57,667: INFO: 1687262790: artifacts/data_ingestion/data.zip download! with following info: 
Connection: close
Content-Length: 23329
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "c69888a4ae59bc5a893392785a938ccd4937981c06ba8a9d6a21aa52b4ab5b6e"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: DA82:3B17D0:910A7:A12FF:6A2324DF
Accept-Ranges: bytes
Date: Fri, 05 Jun 2026 19:34:56 GMT
Via: 1.1 varnish
X-Served-By: cache-fra-eddf8230228-FRA
X-Cache: MISS
X-Cache-Hits: 0
X-Timer: S1780688096.922483,VS0,VE135
Vary: Authorization,Accept-Encoding
Access-Control-Allow-Origin: *
Cross-Origin-Resource-Policy: cross-origin
X-Fastly-Request-ID: ff9c939f877fcb7ee111832de39aed27a047b0a2
Expires: Fri, 05 Jun 2026 19:39:56 GMT
Source-Age: 0

]
artifacts/data_ingestion/data.zip
